In [ ]:

save_path = '/omics/odcf/analysis/OE0260_projects/spatial/jakob/registration_xenium_macsima_amplify/reg_sharing/'

# bsub -R "rusage[mem=200G]" -q long -oo /omics/odcf/analysis/OE0260_projects/spatial/jakob/registration_xenium_macsima_amplify/reg_sharing/Copy_data.log \
# bash -c 'source /omics/groups/OE0260/internal/jakob/miniforge-pypy3/bin/activate && conda activate /home/j548r/mamba_envs/spatial && ipython -c "%run /omics/odcf/analysis/OE0260_projects/spatial/jakob/registration_xenium_macsima_amplify/reg_sharing/Copy_data.ipynb"'

# ─────────────────────────────────────────────────────────────────────────────
# Copy_data.ipynb — extract everything a self-contained registration run needs
#
# This notebook produces a minimal, shareable data package under data/ that
# lets 00_match_rois_cores.ipynb run without access to the original:
#   • all_cells_annotated.h5ad   (full multi-slide annotated dataset)
#   • Xenium output folder       (morphology focus, full transcript parquet, …)
#
# Outputs written per Xenium position / MACSima ROI pair:
#   data/annot_xenium/adata_position_{POS}.h5ad        – cell-type annotations
#   data/raw_xenium/dx_position_{POS}.h5ad             – Xenium cell table + spatial coords
#   data/raw_xenium/bbox_position_{POS}.npy            – ROI bounding box (4 floats)
#   data/raw_xenium/transform_factor_position_{POS}.npy – µm-per-pixel scale factor
#   data/raw_xenium/mf_roi_position_{POS}.tif          – preprocessed morphology focus crop (SIFT input)
#   data/raw_xenium/mf_full_roi_position_{POS}.tif     – full-res morphology focus crop (SpatialData ref)
#   data/raw_xenium/cell_boundaries_position_{POS}.parquet     – cell polygons
#   data/raw_xenium/nucleus_boundaries_position_{POS}.parquet  – nucleus polygons
#   data/raw_xenium/transcripts_position_{POS}.parquet – transcripts cropped to ROI
#   data/macsima/ROI{ROI}/                             – MACSima channel images
# ─────────────────────────────────────────────────────────────────────────────

import os
import sys
import shutil
import numpy as np
import tifffile
import scanpy as sc
import geopandas as gpd
import dask
dask.config.set({'dataframe.query-planning': False})

sys.path.append('/omics/odcf/analysis/OE0260_projects/spatial/jakob/registration_xenium_macsima_amplify/')
from xenium_macsima_core_registration import XeniumMacsimaRegistration
from spatialdata.transformations import get_transformation

# ── Position→ROI mapping (established by comparing TMA slide layouts) ──────
position_roi_map = {
    '1.0' : 40,
    '2.0' : None,
    '3.0' : 36,
    '4.0' : 34,
    '5.0' : 32,
    '6.0' : 39,
    '7.0' : 37,
    '8.0' : 35,   # +38
    '9.0' : 33,
    '10.0' : 31,
    '11.0' : 30,
    '12.0' : 28,
    '13.0' : 26,
    '14.0' : 24,
    '15.0' : 22,
    '16.0' : 29,
    '17.0' : 27,
    '18.0' : 25,
    '19.0' : 23,
    '20.0' : 21,
    '21.0' : 20,
    '22.0' : 18,
    '23.0' : 16,
    '24.0' : 14,
    '25.0' : 12,
    '26.0' : 19,
    '27.0' : 17,
    '28.0' : 15,
    '29.0' : 13,
    '30.0' : 11,
    '31.0' : 10,
    '32.0' : 8,
    '33.0' : 6,
    '34.0' : 4,
    '35.0' : 2,
    '36.0' : 9,
    '37.0' : 7,
    '38.0' : 5,
    '39.0' : 3,
    '40.0' : 1,
}

print("✓ Imports successful")
print(f"  Positions to process: {len(position_roi_map)}  "
      f"(skip None: {sum(v is None for v in position_roi_map.values())})")


In [ ]:

# ── Fixed parameters ──────────────────────────────────────────────────────────
adata_path   = '/omics/odcf/analysis/OE0260_projects/spatial/jakob/processed_data/all_cells_annotated.h5ad'
SLIDE_NAME   = 'TMAN02-1-1'
SCALE_FACTOR = 3    # down-scale factor used throughout the pipeline
XENIUM_FOLDER = (
    '/omics/odcf/analysis/OE0260_projects/spatial/Xenium_amplify/project057/'
    '20250703__072639__057_D170_5K_Run01_2025-07-03/'
    'output-XETG00423__0059691__TMAN02-1-1__20250703__072704/'
)
MACSIMA_BASE = (
    '/omics/odcf/analysis/OE0260_projects/spatial/MACSima_amplify/'
    'MICS_2025-07-14_12-00-01_Run154TMA1JRSS_Run154TMA1JRSS/'
    '20250731_preprocessing_new/Run154TMA1JRSS/R1/A1/'
)

# ── Choose one position to process interactively (change as needed) ──────────
# When running the full batch via bsub, loop over all position_roi_map items.
POSITION  = '1.0'
ROI_INDEX = position_roi_map[POSITION]

macsima_folder = os.path.join(MACSIMA_BASE, f'ROI{ROI_INDEX}/')

# Identify the cycle-0 DAPI file (single-exposure, not background frame)
dapi_candidates = [
    f for f in os.listdir(macsima_folder)
    if 'DAPI' in f and 'C-000' in f and '_S_' in f
]
assert len(dapi_candidates) == 1, (
    f"Expected one cycle-0 DAPI file for ROI {ROI_INDEX}, found: {dapi_candidates}"
)

PATHS = {
    'h5ad'          : adata_path,
    'xenium_folder' : XENIUM_FOLDER,
    'macsima_image' : os.path.join(macsima_folder, dapi_candidates[0]),
    'macsima_folder': macsima_folder,
}

print(f"Processing Xenium position {POSITION}  →  MACSima ROI {ROI_INDEX}")
print(f"  MACSima DAPI : {dapi_candidates[0]}")


In [ ]:

# ── Step 1: extract per-core annotations from the full dataset ────────────────
# load_annotated_data() reads the complete multi-slide h5ad, filters to
# SLIDE_NAME + is_in_core + POSITION, and stores the result on registration.adata.
registration = XeniumMacsimaRegistration(scale_factor=SCALE_FACTOR)
print("Loading annotated single-cell data (reads full h5ad) …")
adata = registration.load_annotated_data(
    h5ad_path=PATHS['h5ad'],
    slide_name=SLIDE_NAME,
    position=POSITION,
)
print(f"✓ {adata.n_obs} cells in position {POSITION}")


In [ ]:

# Keep only the columns needed by the registration notebook.
# This strips large .uns (e.g. UMAP embeddings, colour maps) and any
# observation columns that are not needed downstream, shrinking the file size.
del adata.uns
adata.obs = adata.obs[[
    'transcripts_nuclear', 'unique_transcripts_nuclear',
    'nucleus_area', 'nucleus_centroid_x', 'nucleus_centroid_y',
    'cell_area', 'cell_centroid_x', 'cell_centroid_y',
    'slide', 'core', 'obs_name_orig',
    'cell_type_lvl_1', 'cell_type_lvl_2', 'cell_type_lvl_3',
]]

# Set obs_names to match the Xenium cell IDs used to index boundaries.
# Without this, downstream notebooks loading this file would have obs_names
# that don't match boundary indices, causing transform_boundaries() to
# silently skip all polygons.
# Clear index.name afterwards to avoid an h5ad write conflict (the index name
# 'obs_name_orig' would collide with the column of the same name).
adata.obs_names = adata.obs['obs_name_orig'].astype(str)
adata.obs.index.name = None

print(f"✓ Trimmed obs to {len(adata.obs.columns)} columns")


In [ ]:

# Save the trimmed annotated data for this core.
# The registration notebook reads this file directly without loading the
# complete multi-slide dataset.
save_path = '/omics/odcf/analysis/OE0260_projects/spatial/jakob/registration_xenium_macsima_amplify/reg_sharing/'
os.makedirs(f"{save_path}/data/annot_xenium", exist_ok=True)
adata.write_h5ad(f"{save_path}/data/annot_xenium/adata_position_{POSITION}.h5ad")
print(f"✓ Saved annotations → {save_path}/data/annot_xenium/adata_position_{POSITION}.h5ad")


In [ ]:

# ── Step 2: load Xenium SpatialData and align cell tables ─────────────────────
# This is the slow step (~1–2 min): reads the full Xenium output folder.
# After alignment, xenium_data holds cell/nucleus boundaries and transcripts;
# dx holds the Xenium expression table filtered to this core's cells.
print("Loading Xenium spatial data (reads full output folder) …")
xenium_data = registration.load_xenium_data(PATHS['xenium_folder'])
print(f"✓ Transform factor (µm per pixel): {registration.transform_factor:.4f}")

print("\nAligning cell tables …")
adata_filtered, dx = registration.align_data_tables()
print(f"✓ {adata_filtered.n_obs} cells retained after alignment")


In [ ]:

# ── Step 3: compute bounding box before saving dx ─────────────────────────────
# calculate_roi_bbox() converts cell centroids to down-scaled image coordinates
# and sets dx.obsm['spatial_transformed'].  We must call this BEFORE saving dx
# so that the registration notebook can skip this step entirely.
print("Computing ROI bounding box …")
bbox = registration.calculate_roi_bbox(dx)
print(f"✓ Bounding box (down-scaled px): "
      f"x=[{bbox[0]:.0f}, {bbox[2]:.0f}]  "
      f"y=[{bbox[1]:.0f}, {bbox[3]:.0f}]")


In [ ]:
# ── Step 4: save Xenium cell table, bbox, and scale factor ────────────────────
os.makedirs(f"{save_path}/data/raw_xenium", exist_ok=True)

# After align_data_tables(), obs.index is named 'obs_name_orig' because it was
# set from that column.  AnnData's h5ad writer raises a ValueError when the
# index name matches a column name whose values differ from the index values.
# Resetting the index name to None resolves the conflict without losing data.
adata_filtered.obs.index.name = None
dx.obs.index.name = None

# dx is saved AFTER calculate_roi_bbox so obsm['spatial_transformed'] is included.
adata_filtered.write_h5ad(
    f"{save_path}/data/raw_xenium/adata_filtered_position_{POSITION}.h5ad"
)
dx.write_h5ad(
    f"{save_path}/data/raw_xenium/dx_position_{POSITION}.h5ad"
)
# Bounding box: (x0, y0, x1, y1) in down-scaled image pixel units.
np.save(f"{save_path}/data/raw_xenium/bbox_position_{POSITION}.npy", np.array(bbox))
# Scale factor that converts Xenium µm to Xenium pixel coordinates.
np.save(
    f"{save_path}/data/raw_xenium/transform_factor_position_{POSITION}.npy",
    np.array([registration.transform_factor])
)

print(f"✓ dx                → {save_path}/data/raw_xenium/dx_position_{POSITION}.h5ad")
print(f"✓ adata_filtered    → {save_path}/data/raw_xenium/adata_filtered_position_{POSITION}.h5ad")
print(f"✓ bbox              → {save_path}/data/raw_xenium/bbox_position_{POSITION}.npy")
print(f"✓ transform_factor  → {save_path}/data/raw_xenium/transform_factor_position_{POSITION}.npy")


In [ ]:

# ── Step 5: save morphology focus crops ───────────────────────────────────────
# The registration notebook needs two versions of the Xenium nuclear stain:
#
#   mf_roi         – preprocessed (CLAHE + uint8, down-scaled) — input for SIFT
#   mf_full_roi    – full-resolution crop of the same region — stored in SpatialData
#
# Saving these crops means the registration notebook never needs to open the
# large (~GB) morphology_focus_0000.ome.tif file.
print("Loading morphology focus image …")
mf_img = registration.load_morphology_focus(PATHS['xenium_folder'])
print(f"✓ Full image shape (C, Y, X): {mf_img.shape}")

# Preprocessed crop (SIFT input)
mf_roi = registration.extract_roi_from_morphology(mf_img, bbox)
tifffile.imwrite(f"{save_path}/data/raw_xenium/mf_roi_position_{POSITION}.tif", mf_roi)
print(f"✓ mf_roi (SIFT input)     : {mf_roi.shape}  "
      f"→ {save_path}/data/raw_xenium/mf_roi_position_{POSITION}.tif")

# Full-resolution crop (structural reference in the final SpatialData)
full_mf = mf_img[0, ...].compute()
full_mf_roi = full_mf[
    int(bbox[1] * SCALE_FACTOR): int(bbox[3] * SCALE_FACTOR),
    int(bbox[0] * SCALE_FACTOR): int(bbox[2] * SCALE_FACTOR),
]
tifffile.imwrite(f"{save_path}/data/raw_xenium/mf_full_roi_position_{POSITION}.tif", full_mf_roi)
print(f"✓ mf_full_roi (full-res)  : {full_mf_roi.shape}  "
      f"→ {save_path}/data/raw_xenium/mf_full_roi_position_{POSITION}.tif")


In [ ]:

# ── Step 6: save cell and nucleus boundaries (GeoParquet) ─────────────────────
# After align_data_tables() the boundaries are already filtered to only the
# cells in this core.  Saving them as GeoParquet (via geopandas) preserves the
# polygon geometries compactly without requiring the full Xenium spatialdata.
cell_boundaries = registration.xenium_data['cell_boundaries']
nuc_boundaries  = registration.xenium_data['nucleus_boundaries']

cell_boundaries.to_parquet(
    f"{save_path}/data/raw_xenium/cell_boundaries_position_{POSITION}.parquet"
)
nuc_boundaries.to_parquet(
    f"{save_path}/data/raw_xenium/nucleus_boundaries_position_{POSITION}.parquet"
)
print(f"✓ cell_boundaries   : {len(cell_boundaries)} polygons  "
      f"→ data/raw_xenium/cell_boundaries_position_{POSITION}.parquet")
print(f"✓ nucleus_boundaries: {len(nuc_boundaries)} polygons  "
      f"→ data/raw_xenium/nucleus_boundaries_position_{POSITION}.parquet")


In [ ]:

# ── Step 7: save transcripts filtered to this ROI ─────────────────────────────
# The full Xenium transcript parquet covers the entire slide.  We filter to the
# bounding box of this core (in the original Xenium coordinate system so that
# the coordinate-shift logic in add_table_and_transcripts() still works) and
# save just those rows as a standalone parquet file.
min_x = bbox[0] * SCALE_FACTOR / registration.transform_factor
min_y = bbox[1] * SCALE_FACTOR / registration.transform_factor
max_x = bbox[2] * SCALE_FACTOR / registration.transform_factor
max_y = bbox[3] * SCALE_FACTOR / registration.transform_factor

transcripts_full = registration.xenium_data['transcripts']
transcripts_roi  = transcripts_full[
    (transcripts_full['x'] >= min_x) &
    (transcripts_full['x'] <= max_x) &
    (transcripts_full['y'] >= min_y) &
    (transcripts_full['y'] <= max_y)
].compute()   # materialise Dask → pandas DataFrame

transcripts_roi.to_parquet(
    f"{save_path}/data/raw_xenium/transcripts_position_{POSITION}.parquet", index=False
)
print(f"✓ Saved {len(transcripts_roi):,} transcripts ({transcripts_roi['feature_name'].nunique()} genes)  "
      f"→ {save_path}/data/raw_xenium/transcripts_position_{POSITION}.parquet")


In [ ]:

# ── Step 8: copy MACSima channel images for this ROI ─────────────────────────
# 00_match_rois_cores.ipynb reads MACSima TIFFs from data/macsima/ROI{ROI}/
# This step copies all .tif files from the original MACSima folder into that
# local directory so the registration notebook is fully self-contained.
import shutil

macsima_src = os.path.join(MACSIMA_BASE, f'ROI{ROI_INDEX}')
macsima_dst = os.path.join(save_path, 'data', 'macsima', f'ROI{ROI_INDEX}')
os.makedirs(macsima_dst, exist_ok=True)

tif_files = [f for f in os.listdir(macsima_src) if f.endswith('.tif')]
for fname in tif_files:
    src_path = os.path.join(macsima_src, fname)
    dst_path = os.path.join(macsima_dst, fname)
    if not os.path.exists(dst_path):
        shutil.copy2(src_path, dst_path)

print(f"✓ MACSima images → {macsima_dst}")
print(f"  Copied {len(tif_files)} TIFF files from ROI {ROI_INDEX}")


In [ ]:

# ── Summary ───────────────────────────────────────────────────────────────────
import glob
files = sorted(glob.glob(f"{save_path}/data/raw_xenium/*_position_{POSITION}.*"))
print(f"\nData package for Xenium position {POSITION} / MACSima ROI {ROI_INDEX}:")
for f in files:
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {os.path.basename(f):<55}  {size_mb:6.1f} MB")
macsima_files = os.listdir(f"{save_path}/data/macsima/ROI{ROI_INDEX}")
print(f"\n  {save_path}/data/macsima/ROI{ROI_INDEX}/  ({len(macsima_files)} channel TIFFs)")
annot_size_mb = os.path.getsize(f"{save_path}/data/annot_xenium/adata_position_{POSITION}.h5ad") / 1e6
print(f"  {save_path}/data/annot_xenium/adata_position_{POSITION}.h5ad  {annot_size_mb:.1f} MB")
print("\n✓ Data package complete — ready for 00_match_rois_cores.ipynb")
